# 01 — Inference mechanics

Runs one greedy and one seeded sampled generation through the INFERX use cases and prints labelled metrics.

**How to run:** install your own Jupyter (it is not an INFERX dependency), then from `backend/` run `uv sync --extra ml` and start the notebook with that environment so `import app` works. Outputs are intentionally not committed.

Every value below carries a provenance label: **measured** (adapter-observed at runtime), **derived** (computed from model config) or **conceptual** (explanation only, no number).

In [ ]:
from app.application.dto.metrics import RunMetrics
from app.bootstrap.container import build_container
from app.domain.inference.generation import GenerationConfig
from app.infrastructure.config.settings import Settings

c = build_container(Settings())
c.load_model.execute(c.settings.to_model_spec())

## Greedy run (temperature 0)
Timings are **measured** by the adapter; token counts are **measured**; the KV-cache size is **derived** from model config.

In [ ]:
prompt = "The capital of France is"
result = c.generate_text.execute(prompt, GenerationConfig(max_new_tokens=16))
kv = c.estimate_kv_cache.execute(result.prompt_tokens + result.output_tokens)
print(result.text)
RunMetrics.from_result(result, kv).to_dict()

## Seeded sampling run
Same prompt with `temperature=0.8`, `top_k=40`, `top_p=0.9`, `seed=7`. Re-running the cell gives the same text on the same hardware and software.

In [ ]:
cfg = GenerationConfig(max_new_tokens=16, temperature=0.8, top_k=40, top_p=0.9, seed=7)
sampled = c.generate_text.execute(prompt, cfg)
print(sampled.text)
RunMetrics.from_result(sampled).to_dict()

## Reading the numbers
- **TTFT** covers tokenization, prefill and first-token selection (Transformers exposes no separate prefill timer); it excludes lock/queue wait.
- **TPOT / decode TPS** are `null` when only one token was generated.
- One run is not a benchmark. See `docs/inference-mechanics.md`.

In [ ]:
c.execution.unload()